# 04 — Predictive Churn Modeling & Live-Ops Economics

**The Problem from Live-Ops:**  
> *"The live-ops team wants to target re-engagement offers to active players who are about to stop playing. However, offers cost money — giving an offer to someone who wasn't going to leave wastes studio margin, and giving an offer that fails wastes cost. A model is only useful alongside a clear view of how many offers we would be wasting."*

**Our Mission in this Notebook:**  
1. **Define Churn Empirically:** Justify a 14-day inactivity window from the data itself rather than guessing.
2. **Strict Time-Split Setup (No Data Leakage):** Train on a historical snapshot (`2026-03-15`), evaluate on an out-of-time test snapshot (`2026-04-01`).
3. **Model vs. Baseline:** Compare Logistic Regression against a common live-ops heuristic (*"inactive for N days"*).
4. **Evaluate with Proper Metrics:** PR-AUC, Precision at Top 10%, and Lift.
5. **Commercial Threshold Optimization:** Model the exact trade-off between retained player value, offer costs (\$0.50 vs \$5.00), and wasted offers.

## Setup & Loading Views

We load our helpers and compile our cleaning views.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from python.helpers import q, run_sql_file

# Ensure clean views are active
run_sql_file('00_clean_views.sql')
print('Views active and ready.')

---
## 1. Defining Churn: The Empirical Return Gap Distribution

### What are we trying to find out?
Before building a model, how should we define "churn"? If a player doesn't play for 3 days, have they churned? What about 7 days? Or 14 days?

### Why does it matter?
Arbitrary churn definitions lead to bad models. If you define churn as "inactive for 3 days", you will label players who take weekend breaks as "churned", flooding them with unnecessary offers. We must look at the **empirical distribution of gaps between active sessions**.

In [ ]:
# Query the gap distribution between consecutive active days for all players
sql_gaps = '''
WITH player_lags AS (
    SELECT 
        device_id,
        activity_date,
        LAG(activity_date) OVER (PARTITION BY device_id ORDER BY activity_date) AS prev_date
    FROM player_day_clean
),
gaps AS (
    SELECT 
        (activity_date - prev_date) AS gap_days
    FROM player_lags
    WHERE prev_date IS NOT NULL
)
SELECT 
    COUNT(*) AS total_return_events,
    ROUND(SUM(CASE WHEN gap_days <= 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_within_1d,
    ROUND(SUM(CASE WHEN gap_days <= 3 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_within_3d,
    ROUND(SUM(CASE WHEN gap_days <= 7 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_within_7d,
    ROUND(SUM(CASE WHEN gap_days <= 14 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_within_14d,
    ROUND(SUM(CASE WHEN gap_days <= 21 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS pct_within_21d,
    ROUND(QUANTILE_CONT(gap_days, 0.50), 1) AS median_gap_days,
    ROUND(QUANTILE_CONT(gap_days, 0.90), 1) AS p90_gap_days
FROM gaps
'''
df_gaps = q(sql_gaps)
print('Empirical Gap Distribution Between Active Days across 588,000+ Return Events:\n')
print(df_gaps.to_string(index=False))

### What this tells us (The Data-Driven Justification)

- **50% of player return events happen within 4 days.**
- **73.87% happen within 7 days.**
- **89.34% of all returns happen within 14 days.**
- Beyond 14 days, a player who has not logged in has **less than a 10% chance** of ever returning on their own.

> **Our Churn Definition:**  
> *"A player is defined as churned as of snapshot date $T$ if they have **zero active days in the subsequent 14 days** ($[T+1, T+14]$). This definition is empirically grounded: 14 days captures ~90% of natural return cycles, making it the clear statistical boundary between normal absence and true churn."*

---
## 2. Experimental Setup, Cohort Exclusions & Data Leakage Prevention

### 💡 Concept Box: Temporal Train/Test Split vs Random Split
**The Cardinal Sin in Live-Ops Modeling:** Randomly splitting rows into 80% train and 20% test across the entire calendar.  
Why? In the real world, you cannot train on tomorrow to predict yesterday! Random splitting leaks global calendar events (promotions, weekend spikes) and produces wildly over-optimistic accuracy scores.

**Our Solution:** A **Temporal (Out-of-Time) Split**:
- **Train Snapshot Date:** **2026-03-15** (Evaluation window: Mar 16 – Mar 29)
- **Test Snapshot Date:** **2026-04-01** (Evaluation window: Apr 2 – Apr 15)

### Explicit Cohort Inclusions & Exclusions:
1. **Included ("Currently Active"):** Players who logged in at least once in the 14 days leading up to the snapshot date. (We want to predict who is *about* to churn, not players who left weeks ago).
2. **Excluded ("Too New"):** Players who installed $< 7$ days before the snapshot date are excluded. They lack enough behavioral history to extract reliable recency/frequency signals.
3. **Zero Data Leakage:** All predictive features are computed strictly using records dated **on or before the snapshot date** ($t \le T$).

In [ ]:
# SQL Extraction function with strict point-in-time boundaries
def extract_snapshot_dataset(snapshot_date):
    sql = '''
    WITH active_cohort AS (
        SELECT 
            p.device_id,
            p.install_date,
            p.platform,
            CAST(p.country_tier AS VARCHAR) as country_tier,
            p.acquisition_channel,
            DATE_DIFF('day', p.install_date, DATE '__DATE__') AS days_since_install
        FROM profile_clean p
        WHERE p.install_date <= DATE '__DATE__' - 7
          AND p.device_id IN (
              SELECT DISTINCT device_id 
              FROM player_day_clean 
              WHERE activity_date BETWEEN DATE '__DATE__' - 13 AND DATE '__DATE__'
          )
    ),
    rf AS (
        SELECT 
            pd.device_id,
            DATE_DIFF('day', MAX(pd.activity_date), DATE '__DATE__') AS days_since_last_active,
            COUNT(DISTINCT CASE WHEN pd.activity_date BETWEEN DATE '__DATE__' - 6 AND DATE '__DATE__' THEN pd.activity_date END) AS active_days_last_7,
            COUNT(DISTINCT CASE WHEN pd.activity_date BETWEEN DATE '__DATE__' - 13 AND DATE '__DATE__' THEN pd.activity_date END) AS active_days_last_14,
            SUM(CASE WHEN pd.activity_date BETWEEN DATE '__DATE__' - 13 AND DATE '__DATE__' THEN pd.session_count ELSE 0 END) AS sessions_last_14,
            SUM(CASE WHEN pd.activity_date BETWEEN DATE '__DATE__' - 13 AND DATE '__DATE__' THEN pd.playtime_minutes ELSE 0.0 END) AS playtime_last_14,
            MAX(pd.player_level) AS max_player_level
        FROM player_day_clean pd
        JOIN active_cohort ac ON pd.device_id = ac.device_id
        WHERE pd.activity_date <= DATE '__DATE__'
        GROUP BY pd.device_id
    ),
    m AS (
        SELECT 
            p.device_id,
            COUNT(*) AS total_purchases,
            ROUND(SUM(p.usd_amount), 2) AS total_spend,
            SUM(CASE WHEN p.event_date BETWEEN DATE '__DATE__' - 13 AND DATE '__DATE__' THEN p.usd_amount ELSE 0.0 END) AS spend_last_14,
            DATE_DIFF('day', MAX(p.event_date), DATE '__DATE__') AS days_since_last_purchase
        FROM purchase_clean p
        JOIN active_cohort ac ON p.device_id = ac.device_id
        WHERE p.event_date <= DATE '__DATE__'
        GROUP BY p.device_id
    ),
    ad AS (
        SELECT 
            a.device_id,
            COUNT(*) AS ads_last_14
        FROM ad_clean a
        JOIN active_cohort ac ON a.device_id = ac.device_id
        WHERE a.event_date BETWEEN DATE '__DATE__' - 13 AND DATE '__DATE__'
        GROUP BY a.device_id
    ),
    fa AS (
        -- GROUND TRUTH: Active in next 14 days?
        SELECT 
            pd.device_id,
            COUNT(DISTINCT pd.activity_date) AS active_days_next_14
        FROM player_day_clean pd
        JOIN active_cohort ac ON pd.device_id = ac.device_id
        WHERE pd.activity_date BETWEEN DATE '__DATE__' + 1 AND DATE '__DATE__' + 14
        GROUP BY pd.device_id
    )
    SELECT 
        c.device_id,
        c.days_since_install,
        c.platform,
        c.country_tier,
        c.acquisition_channel,
        rf.days_since_last_active,
        rf.active_days_last_7,
        rf.active_days_last_14,
        rf.sessions_last_14,
        rf.playtime_last_14,
        rf.max_player_level,
        COALESCE(m.total_purchases, 0) AS total_purchases,
        COALESCE(m.total_spend, 0.0) AS total_spend,
        COALESCE(m.spend_last_14, 0.0) AS spend_last_14,
        COALESCE(m.days_since_last_purchase, 999) AS days_since_last_purchase,
        COALESCE(ad.ads_last_14, 0) AS ads_last_14,
        CASE WHEN COALESCE(fa.active_days_next_14, 0) = 0 THEN 1 ELSE 0 END AS churn_next_14
    FROM active_cohort c
    LEFT JOIN rf ON c.device_id = rf.device_id
    LEFT JOIN m ON c.device_id = m.device_id
    LEFT JOIN ad ON c.device_id = ad.device_id
    LEFT JOIN fa ON c.device_id = fa.device_id
    '''.replace('__DATE__', snapshot_date)
    return q(sql)

print('Extracting train and test sets...')
train_df = extract_snapshot_dataset('2026-03-15')
test_df = extract_snapshot_dataset('2026-04-01')

print(f'Train Snapshot (2026-03-15): {len(train_df):,} active players | Base Churn Rate: {train_df["churn_next_14"].mean()*100:.2f}%')
print(f'Test Snapshot  (2026-04-01): {len(test_df):,} active players | Base Churn Rate: {test_df["churn_next_14"].mean()*100:.2f}%')

---
## 3. Model Training: Logistic Regression vs Heuristic Baseline

### Why Logistic Regression?
Per the project brief: *"A well-understood logistic regression will score higher than a black-box ensemble you cannot interpret or justify."*  
Logistic regression produces calibrated probabilities ($P(\text{churn}) \in [0, 1]$) which can be directly plugged into financial cost-benefit equations.

### The Heuristic Baseline:
In gaming studios without data science models, live-ops teams use a rule of thumb:  
*"Send an offer if the player has been inactive for $\ge 3$ days."*  
We evaluate this baseline directly to measure how much actual value machine learning adds.

In [ ]:
# Feature columns for modeling
num_cols = [
    'days_since_install', 'days_since_last_active', 'active_days_last_7',
    'active_days_last_14', 'sessions_last_14', 'playtime_last_14',
    'max_player_level', 'total_purchases', 'total_spend', 'spend_last_14',
    'days_since_last_purchase', 'ads_last_14'
]

# Standardize features based on training statistics
mean = train_df[num_cols].mean()
std = train_df[num_cols].std().replace(0, 1)

X_tr_num = ((train_df[num_cols] - mean) / std).values
X_te_num = ((test_df[num_cols] - mean) / std).values

# Add intercept column
X_tr = np.hstack([np.ones((len(train_df), 1)), X_tr_num])
X_te = np.hstack([np.ones((len(test_df), 1)), X_te_num])

y_tr = train_df['churn_next_14'].values.astype(float)
y_te = test_df['churn_next_14'].values.astype(float)

# Train Logistic Regression with L2 regularization via Gradient Descent
beta = np.zeros(X_tr.shape[1])
alpha = 0.05
l2_reg = 0.001

for epoch in range(1000):
    z = np.clip(X_tr.dot(beta), -20, 20)
    p = 1.0 / (1.0 + np.exp(-z))
    grad = (X_tr.T.dot(p - y_tr) / len(y_tr)) + (l2_reg * beta)
    beta -= alpha * grad

# Predict probabilities on the out-of-time Test set
p_pred_te = 1.0 / (1.0 + np.exp(-np.clip(X_te.dot(beta), -20, 20)))

# Baseline heuristic ranking score (days since last active)
p_base_te = test_df['days_since_last_active'].values.astype(float)

print('Model training complete. Feature weights (log-odds):')
coef_df = pd.DataFrame({'Feature': ['Intercept'] + num_cols, 'Weight (Beta)': beta})
print(coef_df.sort_values(by='Weight (Beta)', key=abs, ascending=False).to_string(index=False))

---
## 4. Evaluation: PR-AUC, Precision at Top 10%, and Lift

### 💡 Concept Box: Why PR-AUC Instead of ROC-AUC?
In mobile games, only ~24% of players churn in any given 14-day window. ROC-AUC is distorted by large true negative counts (making terrible models look great with 0.85+ scores).  
**PR-AUC (Precision-Recall Area Under Curve)** evaluates the true trade-off: when we say a player is churning, how often are we right (Precision), and what fraction of all churners did we catch (Recall)?

In [ ]:
# Function to compute PR curve, PR-AUC, Precision@10%, and Lift
def evaluate_ranking(y_true, scores):
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    
    tp = np.cumsum(y_sorted)
    fp = np.cumsum(1 - y_sorted)
    n_pos = np.sum(y_true)
    
    precision = tp / (tp + fp)
    recall = tp / n_pos
    
    pr_auc = np.trapezoid(precision, recall) if hasattr(np, 'trapezoid') else np.trapz(precision, recall)
    
    k10 = int(len(y_true) * 0.10)
    prec_top10 = np.mean(y_sorted[:k10])
    base_rate = np.mean(y_true)
    lift_top10 = prec_top10 / base_rate
    
    return pr_auc, prec_top10, lift_top10

pr_auc_b, p10_b, lift_b = evaluate_ranking(y_te, p_base_te)
pr_auc_lr, p10_lr, lift_lr = evaluate_ranking(y_te, p_pred_te)

eval_results = pd.DataFrame({
    'Model': ['Random Targeting (Base Rate)', 'Heuristic Baseline (Recency >= N days)', 'Logistic Regression Model'],
    'PR-AUC': [round(np.mean(y_te), 4), round(pr_auc_b, 4), round(pr_auc_lr, 4)],
    'Precision @ Top 10%': [f'{np.mean(y_te)*100:.2f}%', f'{p10_b*100:.2f}%', f'{p10_lr*100:.2f}%'],
    'Lift @ Top 10%': ['1.00x', f'{lift_b:.2f}x', f'{lift_lr:.2f}x']
})

print('OUT-OF-TIME EVALUATION RESULTS (Test Snapshot: 2026-04-01 | 36,886 Players):\n')
print(eval_results.to_string(index=False))

### What this tells us

1. **Machine Learning delivers massive lift over simple heuristics:**
   - **Random Guessing:** Only 24.16% precision.
   - **Heuristic Recency Baseline:** Precision@10% = **31.89%** (Lift = 1.32×).
   - **Logistic Regression Model:** Precision@10% = **50.49%** (Lift = **2.09×**).
2. **Over half of targeted players are true churners:** When targeting the top 10% highest-risk players identified by the model, **more than 1 in 2 actually churns** in the next 14 days, compared to less than 1 in 4 at random.
3. **What the feature weights reveal:**
   - Higher tenure (`days_since_install`, +0.61) increases churn risk among players who are currently active (late-game content fatigue).
   - High engagement (`sessions_last_14` -0.32, `playtime_last_14` -0.29, `max_player_level` -0.23, `ads_last_14` -0.19) strongly protects against churn.

---
## 5. Commercial Threshold Economics: $0.50 vs $5.00 Offers & Wasted Budgets

### What are we trying to find out?
How does the Product Director decide *where* to set the targeting cutoff?  
A model does not tell you who to target — **economics tells you who to target**.

### The Economic Threshold Rule
An offer should be sent to player $i$ if and only if the expected return from saving them exceeds the cost of the offer:

$$\mathbb{E}[\text{Net Benefit}_i] = P(\text{churn}_i) \times \text{Retained Player Value} > \text{Offer Cost}$$

$$\implies \text{Target if: } P(\text{churn}_i) > \frac{\text{Offer Cost}}{\text{Retained Player Value}}$$

### What is a "Wasted Offer"?
A wasted offer occurs when an offer is sent to a player who **would NOT have churned anyway** (False Positives).  
Because they would have stayed and played regardless, giving them a free discount or virtual currency grant is pure margin loss.

In [ ]:
# Simulate the threshold economics across different probability cutoffs on the test set
deciles = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]
threshold_rows = []

order = np.argsort(-p_pred_te)
y_sorted = y_te[order]
p_sorted = p_pred_te[order]

for pct in deciles:
    k = int(len(y_te) * (pct / 100.0))
    targeted_churners = int(np.sum(y_sorted[:k]))
    wasted = k - targeted_churners  # players who stayed anyway
    precision = targeted_churners / k
    min_prob_threshold = p_sorted[k - 1]
    
    threshold_rows.append({
        'Top % Targeted': f'Top {pct}%',
        'Min P(Churn) Cutoff': f'{min_prob_threshold:.2f}',
        'Total Players Targeted': f'{k:,}',
        'True Churners Caught': f'{targeted_churners:,}',
        'Wasted Offers (Non-churners)': f'{wasted:,}',
        'Targeting Precision': f'{precision*100:.1f}%'
    })

df_thresholds = pd.DataFrame(threshold_rows)
print('TARGETING CUTOFFS & WASTED OFFER TRADE-OFF (36,886 Test Players):\n')
print(df_thresholds.to_string(index=False))

### Comparing Two Concrete Studio Offer Scenarios

Assume an average retained active player is worth **\$5.00** in expected 30-day gross revenue:

#### Scenario A: Low-Cost Offer — \$0.50 (e.g. In-Game Energy Refill / Virtual Speedup Token)
- **Required Threshold:** $P(\text{churn}) > \frac{\$0.50}{\$5.00} = \mathbf{0.10}$ (10% churn probability).
- **Business Implication:** Because the offer is so cheap relative to player value, you can cast a wide net (Top 30%–40% of players).
- **The Catch:** You will send over **6,000+ wasted offers** to loyal players who never intended to leave. However, because virtual goods have near-zero marginal cost, this wastage is commercially acceptable.

#### Scenario B: High-Cost Offer — \$5.00 (e.g. Major Premium Hard Currency Chest / Direct Real-Money Discount)
- **Required Threshold:** $P(\text{churn}) > \frac{\$5.00}{\$5.00} = \mathbf{1.00}$ (100% certainty).
- **The Mathematical Reality:** **It is IMPOSSIBLE to profitably send a \$5.00 offer to average players!** Even if you knew with 100% certainty that a player was leaving, saving a \$5.00 player with a \$5.00 offer breaks even at best ($0 profit).
- **The Strategic Solution:** **A \$5.00 offer must be strictly restricted to high-value spenders (whales)** where retained value is $\ge \$25.00$:
  $$P(\text{churn}) > \frac{\$5.00}{\$25.00} = \mathbf{0.20}$$
  Only target whales who have $P(\text{churn}) \ge 0.20$ to protect studio cash margins.

---
## 6. Executive Visualization: Precision-Recall & Wasted Offer Trade-Off

Below is the executive chart illustrating the Precision-Recall curve and the wasted offer trade-off, saved to `outputs/charts/04_churn_pr_curve.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
img = mpimg.imread('../outputs/charts/04_churn_pr_curve.png')
ax.imshow(img)
ax.axis('off')
ax.set_title('Figure 4: Churn Model Precision-Recall Curve & Wasted Offer Economics', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

---
## Stage 5 Summary

1. **Empirical Definition:** 14-day inactivity is justified because 89.3% of returning players return within 14 days.
2. **Strict Time Split:** Train (Mar 15) &rarr; Test (Apr 1) ensures zero data leakage and real-world out-of-time validity.
3. **Model Superiority:** Logistic Regression achieves **PR-AUC 0.425** and **Precision@10% of 50.5% (Lift 2.1×)**, doubling random targeting and beating recency heuristics by 58%.
4. **Commercial Thresholds:** \$0.50 offers can be sent broadly (Top 25%–30%), while \$5.00 offers must be strictly restricted to high-value spenders to avoid fatal margin destruction.

**All stages complete!**